In [1]:
import tensorflow as tf

# 시스템에 장착된 GPU 장치 목록 가져오기
gpus = tf.config.list_physical_devices('GPU')
print(gpus)

if gpus:
    try:
        for gpu in gpus:
            # GPU 메모리 동적 할당 설정 (이미지 속 오타 'experimntal' 수정 포함)
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

I0000 00:00:1791695091.705403 1709578 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791695091.753587 1709578 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791695092.683216 1709578 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


W0000 00:00:1791695093.990271 1709578 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.


In [2]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.layers import Conv2D, MaxPooling2D

class MyVGG19:
    @staticmethod
    def build(input_shape=(224,224,3), activation='relu'):
        model = Sequential()

        model.add(Conv2D(64, (3,3), input_shape=input_shape,
                         activation=activation, padding="same"))
        model.add(Conv2D(64, (3,3), activation=activation, padding="same"))
        model.add(MaxPooling2D(pool_size=(2,2)))

        model.add(Conv2D(128, (3,3), activation=activation, padding="same"))
        model.add(Conv2D(128, (3,3), activation=activation, padding="same"))
        model.add(MaxPooling2D(pool_size=(2,2)))

        model.add(Conv2D(256, (3,3), activation=activation, padding="same"))
        model.add(Conv2D(256, (3,3), activation=activation, padding="same"))
        model.add(Conv2D(256, (3,3), activation=activation, padding="same"))
        model.add(Conv2D(256, (3,3), activation=activation, padding="same"))
        model.add(MaxPooling2D(pool_size=(2,2)))

        model.add(Conv2D(512, (3,3), activation=activation, padding="same"))
        model.add(Conv2D(512, (3,3), activation=activation, padding="same"))
        model.add(Conv2D(512, (3,3), activation=activation, padding="same"))
        model.add(Conv2D(512, (3,3), activation=activation, padding="same"))
        model.add(MaxPooling2D(pool_size=(2,2)))

        model.add(Conv2D(512, (3,3), activation=activation, padding="same"))
        model.add(Conv2D(512, (3,3), activation=activation, padding="same"))
        model.add(Conv2D(512, (3,3), activation=activation, padding="same"))
        model.add(Conv2D(512, (3,3), activation=activation, padding="same"))
        model.add(MaxPooling2D(pool_size=(2,2)))

        model.add(Flatten())
        model.add(Dense(4096, activation=activation))
        model.add(Dense(4096, activation=activation))
        model.add(Dense(1000, activation='softmax'))

        return model

In [3]:
from tensorflow.keras.applications.vgg19 import VGG19
vgg = VGG19() # model & weights
weights = vgg.get_weights()

model = MyVGG19().build()
model.set_weights(weights)

W0000 00:00:1791695094.044770 1709578 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1791695094.160645 1709578 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 29802 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5090, pci bus id: 0000:c2:00.0, compute capability: 12.0a
/root/workspace/.venv/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [5]:
from tensorflow.keras.preprocessing import image
img = image.load_img("sample.jpg", target_size=(224,224))
img_data = image.img_to_array(img)
print("before reshape:", img_data.shape)
import numpy as np
img_data = img_data[np.newaxis, ...]
print("after reshape:", img_data.shape)

pred = model.predict(img_data)
from tensorflow.keras.applications.vgg19 import decode_predictions
print(decode_predictions(pred, top=3),"end")

before reshape: (224, 224, 3)
after reshape: (1, 224, 224, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
[[('n02099712', 'Labrador_retriever', np.float32(0.8591977)), ('n02104029', 'kuvasz', np.float32(0.08797705)), ('n02092339', 'Weimaraner', np.float32(0.014393396))]] end
